# Tutorial 2 - Login And Persona Verification

This tutorial explains credential loading, Persona verification, cookie storage, and SMTP notification without contacting the live BRAIN API.


## 1. Credentials File Format

`brain-sim` accepts either a JSON list `[email, password]` or an object with `email` and `password`. Do not commit real credentials.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


In [ ]:
from brain_sim.auth import BrainAuth, load_credentials
from brain_sim.models import AuthChallenge
from brain_sim.notify import MemoryNotifier

login_run_dir = RUNS_DIR / "tutorial_02_login"
shutil.rmtree(login_run_dir, ignore_errors=True)
login_run_dir.mkdir(parents=True)

demo_credentials = login_run_dir / "demo_credentials.json"
demo_credentials.write_text(json.dumps(["researcher@example.com", "REPLACE_ME"]), encoding="utf-8")
email, password = load_credentials(demo_credentials)
print(email, password)


## 2. Persona Verification Link Lifecycle

When BRAIN returns a Persona challenge, the CLI prints the verification link and saves challenge state in `.brain_sim/latest_login_link.json`. This demo writes the same shape into the tutorial run folder.


In [ ]:
challenge = AuthChallenge(
    url="https://inquiry.withpersona.com/verify?inquiry-id=demo-inquiry",
    www_authenticate="persona",
    message="WorldQuant BRAIN requires Persona verification.",
    payload={"inquiry": "demo-inquiry", "verification_url": "https://inquiry.withpersona.com/verify?inquiry-id=demo-inquiry"},
)
challenge_path = login_run_dir / "latest_login_link.json"
challenge_path.write_text(json.dumps(challenge.__dict__, indent=2, sort_keys=True), encoding="utf-8")
print(challenge.url)
print(challenge_path)


## 3. Email Notification Mode

SMTP mode sends the same Persona URL to your email. The live CLI reads `BRAIN_SIM_SMTP_HOST`, `BRAIN_SIM_SMTP_PORT`, `BRAIN_SIM_SMTP_FROM`, `BRAIN_SIM_SMTP_USER`, and `BRAIN_SIM_SMTP_PASSWORD`.


In [ ]:
notifier = MemoryNotifier()
notifier.send_login_link("me@example.com", challenge.url)
notifier.sent_messages[0]


## 4. Cookie Storage And Reload

After successful authentication, cookies are stored in `.brain_sim/cookies.json`. This demo writes a safe fake cookie and confirms `BrainAuth.load_saved_cookies()` can load it.


In [ ]:
cookie_path = login_run_dir / "cookies.json"
cookie_path.write_text(
    json.dumps({"cookies": [{"name": "t", "value": "demo", "domain": "api.worldquantbrain.com", "path": "/", "secure": True, "expires": None}]}, indent=2),
    encoding="utf-8",
)
auth = BrainAuth(cookie_path=cookie_path)
loaded = auth.load_saved_cookies()
print(f"loaded={loaded}")
print(auth.session.cookies.get("t"))


## 5. Live Login Commands

Run these from a terminal when you are ready to authenticate for real:

```bash
brain-sim login --print-link --credentials-file ~/.brain_credentials
brain-sim login --notify-email "me@example.com" --credentials-file ~/.brain_credentials
```

The second command requires SMTP environment variables. Complete Persona in the browser, then run login again to save cookies.
